In [2]:
import pyspark.sql.functions as f
import pyspark.sql.types as t

from gentropy.common.session import Session

Loading BokehJS ...

/Users/dc16/gentropy/.venv/lib/python3.12/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.



In [3]:
session = Session(extended_spark_conf={"spark.driver.memory": "13g"})

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/13 16:55:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/08/13 16:55:33 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/08/13 16:55:33 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


This is the therapeutic area hierarchy:


In [ ]:
therapy_area_hierarchy = {
    "EFO_0001444": "measurement",
    "MONDO_0045024": "cancer or benign tumor",
    "OTAR_0000018": "genetic, familial or congenital disease",
    "EFO_0005741": "infectious disease",
    "OTAR_0000009": "injury, poisoning or other complication",
    "OTAR_0000014": "pregnancy or perinatal disease",
    "MONDO_0024458": "disorder of visual system",
    "EFO_0000319": "cardiovascular disease",
    "EFO_0009605": "pancreas disease",
    "EFO_0010282": "gastrointestinal disease",
    "OTAR_0000017": "reproductive system or breast disease",
    "EFO_0010285": "integumentary system disease",
    "EFO_0001379": "endocrine system disease",
    "OTAR_0000010": "respiratory or thoracic disease",
    "EFO_0009690": "urinary system disease",
    "OTAR_0000006": "musculoskeletal or connective tissue disease",
    "MONDO_0021205": "disorder of ear",
    "EFO_0000540": "immune system disease",
    "EFO_0005803": "hematologic disease",
    "EFO_0000618": "nervous system disease",
    "MONDO_0002025": "psychiatric disorder",
    "OTAR_0000020": "nutritional or metabolic disease",
    "EFO_0003765": "sign or symptom",  # Not a therapeutic area - is descendant of phenotype
    # "EFO_0000651": "phenotype",
    # "GO_0008150":  "biological process",
    # "EFO_0002571":  "medical procedure",
    # "EFO_0005932": "animal disease",
}

In [4]:
studies = session.spark.read.parquet(
    "gs://open-targets-data-releases/25.06/output/study"
)
rescaled_beta = session.spark.read.parquet(
    "gs://genetics-portal-dev-analysis/ss60/gentropy-manuscript/chapters/variant-effect-prediction/25.07/lead_variant_effect"
)

25/08/13 16:55:34 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-google-hadoop-file-system.properties,hadoop-metrics2.properties


# Study-Index with therapeutic areas


In [8]:
# This udf extracts the FIRST therapeutic area, as per hierarchy list, for each diseaseId
@f.udf(t.StringType())
def get_first_matching_therapeutic_area(therapeutic_areas_list):
    if therapeutic_areas_list is None:
        return None
    for ta in therapy_area_hierarchy:
        if ta in therapeutic_areas_list:
            return ta
    return None


# These lines create a dictionary of diseaseId to primary therapeutic area
efo_ta = (
    session.spark.read.parquet(
        "gs://open-targets-data-releases/25.06/output/disease/disease.parquet"
    )
    .select(
        "id",
        "ancestors",
    )
    .withColumn(
        "primaryTherapeuticArea",
        get_first_matching_therapeutic_area(f.col("ancestors")),
    )
    .withColumn(
        "primaryTherapeuticArea",
        f.when(f.col("primaryTherapeuticArea").isNull(), f.lit("other")).otherwise(
            f.col("primaryTherapeuticArea")
        ),
    )
    .join(
        studies.select(f.explode("diseaseIds").alias("efo")),
        f.col("id") == f.col("efo"),
        "semi",
    )
)
efo_ta_lookup = efo_ta.select("id", "primaryTherapeuticArea").collect()
efo_ta_dict = {row["id"]: row["primaryTherapeuticArea"] for row in efo_ta_lookup}


# This udf takes a diseaseIds arrays and creates an array of mapped therapeutic areas
@f.udf(t.ArrayType(t.StringType()))
def map_efos_to_therapeutic_areas(efo_ids):
    if efo_ids is None:
        return None
    lookup_dict = efo_ta_dict
    mapped_areas = []
    for efo_id in efo_ids:
        mapped_areas.append(lookup_dict.get(efo_id, None))
        mapped_areas = list(set(area for area in mapped_areas if area is not None))
    return mapped_areas

In [9]:
gwas = (
    studies.filter(f.col("studyType") == "gwas")
    .withColumn(
        "mappedTherapeuticAreas", map_efos_to_therapeutic_areas(f.col("diseaseIds"))
    )
    .withColumn(
        "measurement", f.array_contains("mappedTherapeuticAreas", "EFO_0001444")
    )
    .withColumn(
        "binaryLessCases",
        f.when(f.col("nCases") < f.col("nControls"), True).otherwise(False),
    )
    .withColumns(
        {
            "cancerOrBenignTumor": f.when(
                f.array_contains("mappedTherapeuticAreas", "MONDO_0045024"), 1
            ).otherwise(0),
            "infectiousDisease": f.when(
                f.array_contains("mappedTherapeuticAreas", "EFO_0005741"), 1
            ).otherwise(0),
            "pregnancyOrPerinatalDisease": f.when(
                f.array_contains("mappedTherapeuticAreas", "OTAR_0000014"), 1
            ).otherwise(0),
            "disorderOfVisualSystem": f.when(
                f.array_contains("mappedTherapeuticAreas", "MONDO_0024458"), 1
            ).otherwise(0),
            "cardiovascularDisease": f.when(
                f.array_contains("mappedTherapeuticAreas", "EFO_0000319"), 1
            ).otherwise(0),
            "pancreasDisease": f.when(
                f.array_contains("mappedTherapeuticAreas", "EFO_0009605"), 1
            ).otherwise(0),
            "gastrointestinalDisease": f.when(
                f.array_contains("mappedTherapeuticAreas", "EFO_0010282"), 1
            ).otherwise(0),
            "reproductiveSystemOrBreastDisease": f.when(
                f.array_contains("mappedTherapeuticAreas", "OTAR_0000017"), 1
            ).otherwise(0),
            "integumentarySystemDisease": f.when(
                f.array_contains("mappedTherapeuticAreas", "EFO_0010285"), 1
            ).otherwise(0),
            "endocrineSystemDisease": f.when(
                f.array_contains("mappedTherapeuticAreas", "EFO_0001379"), 1
            ).otherwise(0),
            "respiratoryOrThoracicDisease": f.when(
                f.array_contains("mappedTherapeuticAreas", "OTAR_0000010"), 1
            ).otherwise(0),
            "urinarySystemDisease": f.when(
                f.array_contains("mappedTherapeuticAreas", "EFO_0009690"), 1
            ).otherwise(0),
            "musculoskeletalOrConnectiveTissueDisease": f.when(
                f.array_contains("mappedTherapeuticAreas", "OTAR_0000006"), 1
            ).otherwise(0),
            "disorderOfEar": f.when(
                f.array_contains("mappedTherapeuticAreas", "MONDO_0021205"), 1
            ).otherwise(0),
            "immuneSystemDisease": f.when(
                f.array_contains("mappedTherapeuticAreas", "EFO_0000540"), 1
            ).otherwise(0),
            "hematologicDisease": f.when(
                f.array_contains("mappedTherapeuticAreas", "EFO_0005803"), 1
            ).otherwise(0),
            "nervousSystemDisease": f.when(
                f.array_contains("mappedTherapeuticAreas", "EFO_0000618"), 1
            ).otherwise(0),
            "psychiatricDisorder": f.when(
                f.array_contains("mappedTherapeuticAreas", "MONDO_0002025"), 1
            ).otherwise(0),
            "nutritionalOrMetabolicDisease": f.when(
                f.array_contains("mappedTherapeuticAreas", "OTAR_0000020"), 1
            ).otherwise(0),
            "geneticFamilialOrCongenitalDisease": f.when(
                f.array_contains("mappedTherapeuticAreas", "OTAR_0000018"), 1
            ).otherwise(0),
            "injuryPoisoningOrOtherComplication": f.when(
                f.array_contains("mappedTherapeuticAreas", "OTAR_0000009"), 1
            ).otherwise(0),
            "signOrSymptom": f.when(
                f.array_contains("mappedTherapeuticAreas", "EFO_0003765"), 1
            ).otherwise(0),
            "other": f.when(
                f.array_contains("mappedTherapeuticAreas", "other"), 1
            ).otherwise(0),
        }
    )
    .withColumn(
        "totalTherapeuticAreas",
        f.col("cancerOrBenignTumor")
        + f.col("infectiousDisease")
        + f.col("pregnancyOrPerinatalDisease")
        + f.col("disorderOfVisualSystem")
        + f.col("cardiovascularDisease")
        + f.col("pancreasDisease")
        + f.col("gastrointestinalDisease")
        + f.col("reproductiveSystemOrBreastDisease")
        + f.col("integumentarySystemDisease")
        + f.col("endocrineSystemDisease")
        + f.col("respiratoryOrThoracicDisease")
        + f.col("urinarySystemDisease")
        + f.col("musculoskeletalOrConnectiveTissueDisease")
        + f.col("disorderOfEar")
        + f.col("immuneSystemDisease")
        + f.col("hematologicDisease")
        + f.col("nervousSystemDisease")
        + f.col("psychiatricDisorder")
        + f.col("nutritionalOrMetabolicDisease")
        + f.col("geneticFamilialOrCongenitalDisease")
        + f.col("injuryPoisoningOrOtherComplication")
        + f.col("signOrSymptom")
        + f.col("other"),
    )
)

In [ ]:
(
    gwas.filter(f.col("binaryLessCases"))
    .groupBy("measurement")
    .agg(f.count("studyId"))
    .show()
)

+-----------+--------------+
|measurement|count(studyId)|
+-----------+--------------+
|       true|          3572|
|      false|         17788|
+-----------+--------------+



In [11]:
gwas.write.parquet(
    "gs://genetics-portal-dev-analysis/dc16/output/gentropy_paper/gwas_therapeutic_areas",
    mode="overwrite",
)

25/08/08 12:08:37 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


# Qualifying studies


In [5]:
si_ta = session.spark.read.parquet(
    "gs://genetics-portal-dev-analysis/dc16/output/gentropy_paper/gwas_therapeutic_areas"
).cache()

25/08/11 13:06:39 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-google-hadoop-file-system.properties,hadoop-metrics2.properties
25/08/11 13:06:42 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [27]:
qualifying_studies = (
    si_ta.filter(f.col("binaryLessCases"))
    .filter(~f.col("measurement"))
    .filter(f.col("nSamples") >= 1000)
    .filter((f.col("nCases") / f.col("nSamples")) >= 0.001)
    .cache()
)
qualifying_studies.count()

25/08/11 13:17:03 WARN CacheManager: Asked to cache already cached data.


15768

In [ ]:
qualifying_studies = qualifying_studies.filter(
    (~f.col("pubmedId").isin(["40069456"])) | (f.col("pubmedId").isNull())
).cache()
qualifying_studies.count()

15730

25/08/11 18:28:09 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 926601 ms exceeds timeout 120000 ms
25/08/11 18:28:09 WARN SparkContext: Killing executors is not supported by current scheduler.
25/08/11 18:45:19 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$

In [ ]:
qualifying_studies.write.parquet(
    "gs://genetics-portal-dev-analysis/dc16/output/gentropy_paper/qualifying_studies",
    mode="overwrite",
)

# Qualifying measurements


In [16]:
# Removing protein measurements and microbiome descendants
# EFO_0007882 - microbiome
# EFO_0004747 - protein measurement

proteins_and_microbiome = (
    session.spark.read.parquet(
        "gs://open-targets-data-releases/25.06/output/disease/disease.parquet"
    )
    .select("id", "descendants")
    .filter(f.col("id").isin(["EFO_0007882", "EFO_0004747"]))
    .select(f.explode("descendants"))
)
proteins_and_microbiome_ids = [row["col"] for row in proteins_and_microbiome.collect()]
proteins_and_microbiome_ids.extend(["EFO_0007882", "EFO_0004747"])

In [17]:
len(proteins_and_microbiome_ids)

6293

In [18]:
qualifying_measurements = (
    si_ta.filter(f.col("measurement"))
    .filter(~f.col("binaryLessCases"))
    .filter(
        f.size(
            f.array_intersect(f.col("diseaseIds"), f.lit(proteins_and_microbiome_ids))
        )
        == 0
    )
)

In [19]:
qualifying_measurements.count()

61885

In [ ]:
(qualifying_measurements.filter(f.col("pubmedId").isin(["40069456"])).count())

0

In [ ]:
qualifying_measurements = qualifying_measurements.filter(
    (~f.col("pubmedId").isin(["40069456"])) | (f.col("pubmedId").isNull())
)
qualifying_measurements.count()

61885

In [22]:
qualifying_measurements.write.parquet(
    "gs://genetics-portal-dev-analysis/dc16/output/gentropy_paper/qualifying_measurements",
    mode="overwrite",
)

# Qualified credible sets generation


In [ ]:
qualifying_studies = session.spark.read.parquet(
    "gs://genetics-portal-dev-analysis/dc16/output/gentropy_paper/qualifying_studies"
)
replicated_cs = session.spark.read.parquet(
    "gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/list_of_gwas_replicated_CSs.parquet"
)
l2g_feature_matrix = session.spark.read.parquet(
    "gs://open-targets-data-releases/25.06/intermediate/l2g_feature_matrix"
)

In [16]:
features = (
    l2g_feature_matrix.groupBy("studyLocusId")
    .agg(
        f.max("eQtlColocH4Maximum").alias("eqtlH4"),
        f.max("eQtlColocClppMaximum").alias("eqtlCLPP"),
        f.max("pQtlColocH4Maximum").alias("pqtlH4"),
        f.max("pQtlColocClppMaximum").alias("pqtlCLPP"),
        f.max("sQtlColocH4Maximum").alias("sqtlH4"),
        f.max("sQtlColocClppMaximum").alias("sqtlCLPP"),
        f.max("vepMaximum").alias("vep"),
    )
    .join(
        replicated_cs.withColumn("replicated", f.lit(1)),
        "studyLocusId",
        "outer",
    )
    .withColumn(
        "replicated",
        f.when(f.col("replicated").isNull(), 0).otherwise(f.col("replicated")),
    )
    .persist()
)
features.show()

+--------------------+----------+------------+---------+-----------+----------+-----------+----+----------+
|        studyLocusId|    eqtlH4|    eqtlCLPP|   pqtlH4|   pqtlCLPP|    sqtlH4|   sqtlCLPP| vep|replicated|
+--------------------+----------+------------+---------+-----------+----------+-----------+----+----------+
|0005218bc3a62e387...|       0.0|         0.0|      0.0|        0.0|       0.0|        0.0|0.66|         0|
|002462a2da2f7c279...|       0.0|         0.0|      0.0|        0.0|       0.0|        0.0| 0.0|         0|
|00274cac95947bd00...|       0.0| 0.019822257|      0.0|        0.0|       0.0|        0.0| 0.1|         0|
|005bc8624f8dd7f7c...|       1.0|        0.95|      0.0|        0.0|       1.0|       0.95|0.33|         1|
|0064268fb58ddabbb...|       0.0|         0.0|      0.0|        0.0|       0.0|        0.0| 0.0|         0|
|008b97f651f444888...|0.97240156| 0.032121874|      0.0|        0.0|       0.0|        0.0| 0.1|         0|
|009c65f69b7705d4c...|0.9760

In [19]:
(
    rescaled_beta.join(
        qualifying_studies.select("studyId", "nCases", "nControls", "nSamples"),
        "studyId",
        "inner",
    ).count()
)

97513

In [ ]:
first_filter_cs = (
    rescaled_beta.join(
        qualifying_studies.select("studyId", "nCases", "nControls", "nSamples"),
        "studyId",
        "inner",
    )
    .filter(2 * f.col("majorLdPopulationMaf.value") * f.col("nCases") >= 20)
    .filter(f.abs("rescaledStatistics.estimatedBeta") <= 3)
    .persist()
)
first_filter_cs.count()

74563

In [21]:
common_cs = first_filter_cs.filter(f.col("majorLdPopulationMaf.value") > 0.01).persist()
common_cs.count()

69150

In [25]:
rare_cs = (
    first_filter_cs.filter(f.col("majorLdPopulationMaf.value") <= 0.01)
    .join(
        features.filter(
            (f.col("eqtlH4") >= 0.8)
            | (f.col("eqtlCLPP") >= 0.01)
            | (f.col("pqtlH4") >= 0.8)
            | (f.col("pqtlCLPP") >= 0.01)
            | (f.col("sqtlH4") >= 0.8)
            | (f.col("sqtlCLPP") >= 0.01)
            | (f.col("vep") >= 0.66)
            | (f.col("replicated") == 1)
        ),
        "studyLocusId",
        "semi",
    )
    .persist()
)
rare_cs.count()

1832

In [26]:
qualifying_credible_sets = (
    common_cs.unionByName(rare_cs)
)
qualifying_credible_sets.count()

70982

In [ ]:
qualifying_credible_sets.write.parquet(
    "gs://genetics-portal-dev-analysis/dc16/output/gentropy_paper/qualifying_credible_sets",
    mode="overwrite",
)

25/08/08 14:13:20 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 953589 ms exceeds timeout 120000 ms
25/08/08 14:13:20 WARN SparkContext: Killing executors is not supported by current scheduler.
25/08/08 14:13:20 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$

# Qualifying Measurement Credible Sets


In [13]:
qualifying_measurements = session.spark.read.parquet('gs://genetics-portal-dev-analysis/dc16/output/gentropy_paper/qualifying_measurements')

In [14]:
(
    rescaled_beta.join(
        qualifying_measurements.select("studyId", "nSamples"), "studyId", "inner"
    ).count()
)

493847

In [ ]:
first_filter_measurements = (
    rescaled_beta.join(
        qualifying_measurements.select("studyId", "nSamples"), "studyId", "inner"
    )
    .filter(2 * f.col("majorLdPopulationMaf.value") * f.col("nSamples") >= 20)
    .filter(f.abs("rescaledStatistics.estimatedBeta") <= 3)
    .persist()
)
first_filter_measurements.count()

473831

In [17]:
common_measurements = first_filter_measurements.filter(f.col("majorLdPopulationMaf.value") > 0.01).persist()
common_measurements.count()

436706

In [18]:
rare_measurements = (
    first_filter_measurements.filter(f.col("majorLdPopulationMaf.value") <= 0.01)
    .join(
        features.filter(
            (f.col("eqtlH4") >= 0.8)
            | (f.col("eqtlCLPP") >= 0.01)
            | (f.col("pqtlH4") >= 0.8)
            | (f.col("pqtlCLPP") >= 0.01)
            | (f.col("sqtlH4") >= 0.8)
            | (f.col("sqtlCLPP") >= 0.01)
            | (f.col("vep") >= 0.66)
            | (f.col("replicated") == 1)
        ),
        "studyLocusId",
        "semi",
    )
    .persist()
)
rare_measurements.count()

13651

In [20]:
qualifying_measurement_credible_sets = (
    common_measurements.unionByName(rare_measurements)
)
qualifying_measurement_credible_sets.count()

450357

In [22]:
qualifying_measurement_credible_sets.write.parquet(
    "gs://genetics-portal-dev-analysis/dc16/output/gentropy_paper/qualifying_measurement_credible_sets",
    mode="overwrite",
)